# Playground 02 — A fake backend you can poke (waiter, kitchen, menu)

📖 Primer: [docs/00-concepts.md](../docs/00-concepts.md), section 1

A real API call crosses the internet. But the IDEA needs no internet: the frontend sends `(method, path)`, the backend looks at them, and returns `(status_code, data)`. Below, `fake_backend` is the kitchen — one ordinary Python function — and the cells after it are you, playing the waiter. In Phase 4, FastAPI replaces this function with the real thing; the shape of the conversation stays identical.

In [ ]:
# The kitchen's pantry — all the data the backend "owns":
BAGS = {
    "chanel-flap-001": {"brand": "Chanel", "model": "Classic Flap", "price": 9500.0},
    "lv-neverfull-001": {"brand": "Louis Vuitton", "model": "Neverfull MM", "price": 1800.0},
    "hermes-birkin-001": {"brand": "Hermès", "model": "Birkin 30", "price": 22000.0},
}

In [ ]:
def fake_backend(method, path, body=None):
    # The entire 'kitchen'. Takes a request, returns (status_code, data).
    # Compare each branch with the primer's menu table — same 4 endpoints.

    # "show me all the bags"
    if method == "GET" and path == "/bags":
        return 200, list(BAGS.values())

    # "show me ONE bag" — e.g. GET /bags/chanel-flap-001
    if method == "GET" and path.startswith("/bags/"):
        bag_id = path.removeprefix("/bags/")   # the part after /bags/
        if bag_id in BAGS:
            return 200, BAGS[bag_id]
        return 404, {"error": f"no bag with id '{bag_id}'"}   # not found!

    # "here's a new bag, please save it"
    if method == "POST" and path == "/bags":
        if body is None or "id" not in body:
            return 400, {"error": "you must send a bag with an 'id'"}
        BAGS[body["id"]] = {k: v for k, v in body.items() if k != "id"}
        return 201, {"saved": body["id"]}      # 201 = created

    # Anything not on the menu:
    return 404, {"error": f"unknown endpoint: {method} {path}"}


def request(method, path, body=None):
    # You, the waiter: carry an order to the kitchen, print the reply.
    status, data = fake_backend(method, path, body)
    print(f"{method} {path}")
    print(f"-> {status}  {data}")

## The menu in action

Run each order and watch the status code that comes back.

In [ ]:
request("GET", "/bags")                  # read everything

In [ ]:
request("GET", "/bags/chanel-flap-001")  # read one

In [ ]:
request("GET", "/bags/gucci-jackie-001") # doesn't exist -> 404

In [ ]:
request("POST", "/bags",                 # create one -> 201
        body={"id": "coach-tabby-001", "brand": "Coach",
              "model": "Tabby 26", "price": 450.0})

request("GET", "/bags/coach-tabby-001")  # ...and now it exists!

In [ ]:
request("DELETE", "/bags/chanel-flap-001")   # not on the menu -> 404

Status codes you just saw: **200** OK · **201** created · **404** not found · (**500** = "kitchen on fire" only happens when the *backend* has a bug.)

## ✏️ Your turn

In [ ]:
# Exercise 1: PREDICT the status code before you run this. Then run.
request("GET", "/bags/hermes-birkin-001")

In [ ]:
# Exercise 2: a bad POST — no "id". What status comes back, and why?
request("POST", "/bags", body={"brand": "Dior"})

In [ ]:
# Exercise 3: add a new endpoint to fake_backend (scroll up, edit that
# cell, RE-RUN it, then run this cell):
#     GET /bags/count  should return  200, {"count": len(BAGS)}
#
# ⚠️ Order matters: put your branch ABOVE the startswith("/bags/")
# branch — otherwise that branch catches it first and thinks "count"
# is a bag id. (Try it BELOW the branch first and watch that exact
# bug happen!)
request("GET", "/bags/count")

In [ ]:
# Exercise 4 (stretch): support DELETE /bags/{id} in fake_backend —
# remove the bag from BAGS and return 200. That's the D in CRUD, one
# notebook early. Re-run the fake_backend cell, then run this:
request("DELETE", "/bags/chanel-flap-001")
request("GET", "/bags/chanel-flap-001")   # should now be a 404!